### Processing EMTP-RV Parametric Studio outputs

In [ ]:
import pickle
import numpy as np

In [ ]:
from simulation import Simulations

In [ ]:
from utils import create_white_list
from readme import read_me

In [ ]:
# ========================
# *** INPUT PARAMETERS ***
# ========================
# Directory path (EDIT):
simulation_path = ''
# Model file name without extension (EDIT):
name = 'IEEE39_Wind_v5_x'
# Short-circuit duration (EDIT): 
sc_time = 'T100ms'  # 'T100ms' or 'T300ms'

In [ ]:
# List of machine variable names:
varnames = [
    '/Teta_1_SM1',   # rotor angle
    '/Omega_1_SM1',  # rotor speed
    '/PowerAng_SM1', # power angle
    '/Pe_SM1',       # electrical power
    '/vd_SM1',  # d-axis stator voltage
    '/id_SM1',  # d-axis stator current
    '/Ef_SM1',  # EMF voltage (q-axis)
    '/vq_SM1',  # q-axis stator voltage
    '/iq_SM1',  # q-axis stator current
]
# List of machines that are excluded.
exclude = []

# List of bus index values.
buses = np.arange(start=1, stop=30).tolist()
# Buses 30 and 32 to 38 are skipped, since they are
# between the generator and its step-up transformer.
buses.extend([31, 39])  # generator buses with load
# New artificial nodes in the middle of each transmission line.
# There are no voltage measurements at these nodes.
middle_points = np.arange(start=40, stop=73).tolist()
# List of all nodes where short-circuits will be applied.
# A distinction is made between nodes (where SC is applied)
# and buses (where voltage is measured).
nodes = buses + middle_points

# Power system variant.
variant = 'V0'

In [ ]:
# Generate the "white_list" variable.
white_list = create_white_list(varnames, exclude, buses, variant)
white_list

In [ ]:
# Build all simulations.
sims = Simulations(simulation_path, name, white_list)
sims.build_all_simulations()

In [ ]:
nsim = sims.get_nb_simu_tot()
print(f'Total no. of simulations: {nsim}')

# Dictionary keys for simulations which identify 
# SC type and node number of the fault location.
sim_keys = ['SC3-' + 'BUS'+str(k) for k in nodes]
sim_keys.extend(['SC2-' + 'BUS'+str(k) for k in nodes])
sim_keys.extend(['SC1-' + 'BUS'+str(k) for k in nodes])

if nsim != len(sim_keys):
    raise ValueError()

# Name pairs for renaming select columns.
name_pairs = {
    # Wind farm signals.
    'DEV2/P': 'WF/P',
    'DEV2/Q': 'WF/Q',
    'DEV2/V0': 'WF/V0',
    'DEV2/V1': 'WF/V1',
    'DEV2/V2': 'WF/V2',
    'DEV2/I0': 'WF/I0',
    'DEV2/I1': 'WF/I1',
    'DEV2/I2': 'WF/I2',
    'FFC_WP2/Wind_Turbine/PMSG_T_rotor': 'WF/PMSG_T_rotor',
    'FFC_WP2/Wind_Turbine/PMSG_w_rotor': 'WF/PMSG_w_rotor',
    'FFC_WP2/Converter_control/Control/Grid_Ctrl/FRT_flag': 'WF/FRT_flag',
    # Solar park signals.
    'DEV3/P': 'PV/P',
    'DEV3/Q': 'PV/Q',
    'DEV3/V0': 'PV/V0',
    'DEV3/V1': 'PV/V1',
    'DEV3/V2': 'PV/V2',
    'DEV3/I0': 'PV/I0',
    'DEV3/I1': 'PV/I1',
    'DEV3/I2': 'PV/I2',
    'WECC_PVPark_1/Converter_control/Control/GridControl_DLL/FRT_flag': 'PV/FRT_flag',
}

In [ ]:
# Dictionary holding DataFrames of signals from all simulations.
data = {}
data['README'] = read_me
for key, index in zip(sim_keys, range(nsim)):
    # Export signals to DataFrame.
    sim = sims.get_simulation(index)
    sim_df = sim.to_dataframe()
    if variant in ['V1', 'V2', 'V3']:
        sim_df.rename(columns=name_pairs, inplace=True)
    # Assign DataFrame to a simulation key.
    data[key] = sim_df

In [ ]:
# Pickle data to the external file.
file_name = variant + '-' + sc_time + '.pkl'
with open(file=file_name, mode='wb') as fp:
    pickle.dump(data, fp)